# 1-09 상향 이벤트 주가반응 — 전망이 오를 때 주가는 따라 올랐나

**질문 예시:** "최근 상향된 종목들, 과거에 상향됐을 때 주가 반응 어땠어?", "SK하이닉스 이익 전망 오를 때마다 주가 어떻게 움직였어?"

CHECK 원본(`1_기업펀더멘털/09_상향이벤트주가반응`)과 같은 흐름입니다.
1. 시가총액 상위 종목에서 **최근 1개월 영업이익 전망이 오른 종목**을 고르고([1-08](08_revision_screener.ipynb)과 같은 계산)
2. 그 종목들의 **과거 2년 동안 컨센서스가 하루에 3% 이상 오른 날**(상향 이벤트)을 찾아
3. 이벤트 뒤 **+1·+5·+20거래일 수익률과 승률**, 코스피 대비 초과수익을 봅니다.

DeepSearch에서 달라진 점: 컨센서스 이력을 증권사별 추정치(`SearchFirmFundamentalsForecasts`)로 **직접 재구성**합니다. 종목별로 한 번씩 부릅니다.

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/beaten-by-the-market/deepsearch/blob/main/scenarios/1_fundamentals/09_upgrade_price_reaction.ipynb)

## 0. 준비
Colab 왼쪽 🔑(Secrets)에 `DEEPSEARCH_API_KEY`를 등록하고 이 노트북의 액세스를 켠 뒤 아래 두 셀을 실행하세요. 두 번째 셀은 모든 시나리오에 공통인 헬퍼입니다(접어 두어도 됩니다).

In [1]:
# Colab에서만: 한글 차트 폰트 설치 (로컬에서는 건너뜀)
import sys
if "google.colab" in sys.modules:
    !pip -q install koreanize-matplotlib

In [2]:
# ── DeepSearch 헬퍼 (모든 시나리오 노트북 공통) ─────────────────────────────
# Colab: 왼쪽 🔑(Secrets)에 이름 DEEPSEARCH_API_KEY 로 키를 넣고 "노트북 액세스"를 켜세요.
#        키는 "아이디:비밀번호" 형식이든 base64 문자열이든 됩니다. 키 값은 출력하지 않습니다.
# 로컬: 환경변수 DEEPSEARCH_API_KEY / API_KEY 또는 상위 폴더의 .env 를 읽습니다.
import base64, json, os, sys, time
from datetime import date, datetime, timedelta

import pandas as pd
import requests

IN_COLAB = "google.colab" in sys.modules
API_URL = "https://api.deepsearch.com/v1/compute"

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)


def _norm_key(k):
    k = k.strip().strip('"').strip("'")
    return base64.b64encode(k.encode()).decode() if ":" in k else k


def _find_key():
    if IN_COLAB:
        try:
            from google.colab import userdata
            k = userdata.get("DEEPSEARCH_API_KEY")
            if k:
                return _norm_key(k)
        except Exception:
            pass
    for name in ("DEEPSEARCH_API_KEY", "API_KEY"):
        if os.environ.get(name):
            return _norm_key(os.environ[name])
    d = os.path.abspath(os.getcwd())
    while True:
        p = os.path.join(d, ".env")
        if os.path.isfile(p):
            for line in open(p, encoding="utf-8-sig"):
                if "=" in line and not line.strip().startswith("#"):
                    k, v = line.split("=", 1)
                    if k.strip() in ("DEEPSEARCH_API_KEY", "API_KEY") and v.strip():
                        return _norm_key(v)
        if os.path.dirname(d) == d:
            break
        d = os.path.dirname(d)
    from getpass import getpass
    return _norm_key(getpass("DeepSearch API 키: "))


_KEY = _find_key()


class DSError(Exception):
    pass


def ds(query, timeout=180, retries=3):
    """compute 호출 → 응답 JSON. 서버가 돌려준 오류(data.exceptions)는 DSError."""
    last = None
    for i in range(retries):
        try:
            r = requests.post(API_URL, json={"input": query}, timeout=timeout,
                              headers={"Authorization": "Basic " + _KEY})
            body = r.json()
        except Exception as e:  # 네트워크·일시 오류만 재시도
            last = e
            time.sleep(3 * (i + 1))
            continue
        exc = (body.get("data") or {}).get("exceptions") or []
        if exc or r.status_code >= 400:
            msg = "; ".join(f"{x.get('class', '')}: {' '.join(map(str, x.get('args') or [x.get('message', '')]))}"
                            if isinstance(x, dict) else str(x) for x in exc)
            raise DSError(f"{r.status_code} {msg or body}"[:500])
        return body
    raise DSError(f"요청 실패: {last}")


def _pod(body):
    pods = (body.get("data") or {}).get("pods") or []
    return pods[1] if len(pods) >= 2 else {}


def ds_df(query, **kw):
    """DataFrame 결과(재무·시세·경제지표 등) → pandas DataFrame. 결과가 없으면 빈 DataFrame."""
    c = _pod(ds(query, **kw)).get("content") or {}
    data = c.get("data") if isinstance(c, dict) else None
    if not isinstance(data, dict) or not data:
        return pd.DataFrame()
    cols = [x for x in (c.get("columns") or data.keys()) if x in data]
    df = pd.DataFrame({x: data[x] for x in cols})
    for x in ("date", "created_at"):
        if x in df and df[x].dtype == object:
            df[x] = pd.to_datetime(df[x].astype(str).str[:19], errors="coerce")
    return df


def _polarity(x):
    """문서 긍부정: polarity.name, 없으면 customized_ml.kis 모델 값(최근 뉴스는 절반가량이 이쪽에만 있음)."""
    p = x.get("polarity")
    if isinstance(p, dict) and p.get("name"):
        return p["name"]
    k = (((x.get("customized_ml") or {}).get("kis") or {}).get("ip_multitask") or {}).get("polarity") or {}
    return k.get("name")


def doc_search(category, section, query, date_from, date_to, count=100, pages=1):
    """DocumentSearch → (총건수, 문서 DataFrame). category: news/research/company/patent. 기간은 1년 이내.
    종목 지정은 securities.symbol:005930 (접두어 없는 6자리)이 뉴스·리서치·공시 모두에서 가장 정확합니다.
    securities.symbol:KRX:005930 은 키워드 검색으로 바뀌어 건수가 줄고, securities.name: 은 리서치(공식명
    '에스케이하이닉스')·공시(DART명 '현대자동차')에서 놓칩니다(2026-11-25 확인). 여러 종목: securities.symbol:(005930 or 000660)"""
    rows, total = [], 0
    for page in range(1, pages + 1):
        q = (f'DocumentSearch(["{category}"],{json.dumps(section, ensure_ascii=False)},'
             f'{json.dumps(query, ensure_ascii=False)},count={count},page={page},'
             f'date_from={date_from},date_to={date_to})')
        c = _pod(ds(q)).get("content") or {}
        d = c.get("data") if isinstance(c.get("data"), dict) else c
        total = d.get("total_matches") or 0
        docs = d.get("docs") or []
        for x in docs:
            secs = x.get("securities") or []
            rows.append({
                "created_at": pd.to_datetime(str(x.get("created_at"))[:19], errors="coerce"),
                "publisher": x.get("publisher"),
                "title": " ".join(str(x.get("title") or "").split()),
                "stocks": ", ".join(s.get("name", "") for s in secs),
                "symbols": ", ".join(s.get("symbol", "") for s in secs),
                "polarity": _polarity(x),
                "url": x.get("content_url"),
            })
        if len(docs) < count:
            break
    cols = ["created_at", "publisher", "title", "stocks", "symbols", "polarity", "url"]
    return total, pd.DataFrame(rows, columns=cols)


def doc_trends(category, section, query, date_from, date_to, interval="1d"):
    """DocumentTrends → DataFrame(date, count)."""
    q = (f'DocumentTrends(["{category}"],{json.dumps(section, ensure_ascii=False)},'
         f'{json.dumps(query, ensure_ascii=False)},interval="{interval}",'
         f'date_from={date_from},date_to={date_to})')
    c = _pod(ds(q)).get("content") or {}
    d = c.get("data") if isinstance(c.get("data"), dict) else c
    trends = d.get("trends") or []
    b = trends[0].get("buckets", []) if trends else []
    df = pd.DataFrame([{"date": x.get("key_as_string") or x.get("key"), "count": x.get("count", 0)} for x in b])
    if not df.empty:
        df["date"] = pd.to_datetime(df["date"].astype(str).str[:19], errors="coerce")
    return df


def stock_prices(entities, date_from, date_to, columns=None):
    """종목 목록의 일별 시세. entities 예: ["KRX:005930", "KRX:000660"] 또는 "KRX:005930".
    가격은 수정주가, 거래량·거래대금·시가총액은 당일 실제 값. change_rate 는 비율(0.01 = 1%)."""
    ents = [entities] if isinstance(entities, str) else list(entities)
    cols = columns or ["open", "high", "low", "close", "volume", "value", "change_rate", "market_cap"]
    q = (f'GetStockPrices([{", ".join(ents)}], columns={json.dumps(cols)}, '
         f'date_from={date_from}, date_to={date_to})')
    return ds_df(q)


_UNIVERSE = None


def universe():
    """현재 상장 종목 기본정보(GetStockSymbols 전 종목): symbol, name, market, industry_name 등.
    company_type_l1: 1 일반법인, 2 공기업(한국전력·기업은행 등 8종목), 8 ETF·ETN 등."""
    global _UNIVERSE
    if _UNIVERSE is None:
        u = ds_df("GetStockSymbols()")
        _UNIVERSE = u.rename(columns={"entity_name": "name"})[
            ["symbol", "name", "market", "company_type_l1", "industry_id", "industry_name", "date_listed",
             "fiscal_year_end", "conglomerate_id"]]
    return _UNIVERSE.copy()


def all_stocks(day, common_only=True, columns=None):
    """day(또는 그 이전 마지막 거래일)의 전 종목 하루치 시세 + 종목 정보. 한 번 호출로 4,000여 종목.
    common_only=True 이면 일반법인·공기업 보통주만(ETF·ETN·우선주 제외)."""
    day = pd.Timestamp(day).date()
    cols = columns or ["open", "high", "low", "close", "volume", "value", "change_rate", "market_cap"]
    snap = ds_df(f'GetStockPrices(columns={json.dumps(cols)}, '
                 f'date_from={day - timedelta(days=7)}, date_to={day})')
    df = snap.merge(universe().drop(columns=["date"], errors="ignore"), on="symbol", how="left")
    if common_only:
        df = df[df["company_type_l1"].isin(["1", "2"]) & df["symbol"].str.endswith("0")]
    df = df[df["market"].isin(["KOSPI", "KOSDAQ", "KONEX"])]
    if "change_rate" in df:
        df["change_pct"] = df["change_rate"] * 100
    return df.reset_index(drop=True)


def index_prices(date_from, date_to):
    """지수 11종(코스피·코스닥 계열) 일별. change_rate 는 퍼센트(1.94 = 1.94%).
    지수의 value(거래대금)는 백만원, volume(거래량)은 천주 단위로 보입니다(코스닥 2026-11-23 value 4,673,819 ≈ 4.7조)."""
    # 지수를 지정하지 않으면 최근 약 1,300행(≈118거래일)만 오고 앞 기간이 조용히 잘린다 → 140일씩 나눠 받는다
    start, end = pd.Timestamp(date_from), pd.Timestamp(date_to)
    parts = []
    while start <= end:
        stop = min(start + pd.Timedelta(days=139), end)
        parts.append(ds_df(f"GetMarketIndexes(date_from={start.date()}, date_to={stop.date()})"))
        start = stop + pd.Timedelta(days=1)
    df = pd.concat([x for x in parts if not x.empty], ignore_index=True) if parts else pd.DataFrame()
    return df.drop_duplicates(["date", "symbol"]).sort_values(["date", "symbol"]).reset_index(drop=True) if not df.empty else df


def fred(series, date_from, date_to):
    """FRED 시리즈 → 날짜 × 시리즈 넓은 표. 예: fred(["DGS10", "DEXKOUS"], "2026-10-01", "2026-11-24")"""
    ids = ", ".join(json.dumps(s) for s in series)
    df = ds_df(f"QueryFredSeriesData([{ids}], date_from={date_from}, date_to={date_to})")
    if df.empty:
        return df
    val = [c for c in df.columns if c not in ("entity_id", "date")][0]
    df["series"] = df["entity_id"].str.replace("FRED:", "", regex=False)
    wide = df.pivot_table(index="date", columns="series", values=val).sort_index()
    return wide[[s for s in series if s in wide.columns]]  # 요청한 순서대로


def krw(v, digits=1):
    """원 → '138.0조' / '5,027억'."""
    if v is None or pd.isna(v):
        return "-"
    a = abs(v)
    if a >= 1e12:
        return f"{v / 1e12:.{digits}f}조"
    if a >= 1e8:
        return f"{v / 1e8:,.0f}억"
    if a >= 1e4:
        return f"{v / 1e4:,.0f}만"
    return f"{v:,.0f}"


PRICE_NOTICE = ("※ 주가는 수정주가입니다(분할·무상증자·유상증자 소급 반영, 배당 미반영). "
                "거래량·거래대금·시가총액은 당일 실제 값이며 KRX 기준입니다(NXT 미포함).")

# 한글 차트 폰트
import matplotlib.pyplot as plt
try:
    import koreanize_matplotlib  # noqa: F401  (Colab: 위 셀에서 설치)
except ImportError:
    if sys.platform.startswith("win"):
        plt.rcParams["font.family"] = "Malgun Gothic"
    elif sys.platform == "darwin":
        plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False

print("DeepSearch 준비 완료", "(Colab)" if IN_COLAB else "(로컬)")

DeepSearch 준비 완료 (로컬)


In [3]:
ASOF = "2026-11-24"
UNIVERSE_TOP = 40      # 시가총액 상위 몇 종목에서 고를지 (종목당 1회 호출)
PICK = 8               # 최근 상향 상위 몇 종목을 백테스트할지
HIST_YEARS = 2         # 과거 이벤트를 찾을 기간
JUMP = 0.03            # 하루 컨센서스 상승률이 이 이상이면 상향 이벤트
GAP = 20               # 이벤트 사이 최소 간격(거래일) — 같은 재료의 연속 상향을 하나로
HORIZONS = (1, 5, 20)
STALE, LOOK = 90, 30
print(PRICE_NOTICE)

※ 주가는 수정주가입니다(분할·무상증자·유상증자 소급 반영, 배당 미반영). 거래량·거래대금·시가총액은 당일 실제 값이며 KRX 기준입니다(NXT 미포함).


## 1. 추정치 받기 (종목별 1회, 과거 2년 + 90일)

In [4]:
from datetime import timedelta
asof = pd.Timestamp(ASOF)
hist_start = (asof - pd.DateOffset(years=HIST_YEARS) - timedelta(days=STALE)).date()
snap = all_stocks(ASOF)
uni = snap.sort_values("market_cap", ascending=False).head(UNIVERSE_TOP)[["symbol", "name"]]
frames = []
for sym in uni["symbol"]:
    f = ds_df(f'SearchFirmFundamentalsForecasts({sym}, last_only=False, accounting_types="K", '
              f'date_from={hist_start}, date_to={ASOF})')
    if f.empty or not f.notna().any().any():
        continue
    f = f.dropna(axis=1, how="all")
    f["symbol"] = sym
    v = f["csd_operating_income"] if "csd_operating_income" in f else pd.Series(index=f.index, dtype=float)
    f["value"] = v.where(v.notna(), f.get("operating_income"))
    frames.append(f.dropna(subset=["value"])[["symbol", "date", "inst_code", "forecast_date", "value"]])
fc = pd.concat(frames, ignore_index=True)
fc["value"] = fc["value"].astype(float)
print(f"{fc['symbol'].nunique()}종목, 추정치 {len(fc):,}행, 기간 {fc['date'].min().date()} ~ {fc['date'].max().date()}")

39종목, 추정치 4,575행, 기간 2024-08-26 ~ 2026-11-23


## 2. 컨센서스 함수와 최근 리비전
1-08과 같습니다. 시점 t의 컨센서스 = t 이전 90일 안 증권사별 최신 추정치의 평균, 중앙값의 1/3~3배 밖 입력 오류는 제외.

In [5]:
def consensus(df, t):
    d = df[(df["date"] <= t) & (df["date"] > t - timedelta(days=STALE))]
    last = d.sort_values("date").groupby("inst_code").tail(1)["value"]
    med = last.median()
    if len(last) >= 3 and med > 0:
        last = last[(last >= med / 3) & (last <= med * 3)]
    return (last.mean(), len(last)) if len(last) else (float("nan"), 0)

FY_NEXT = f"{asof.year + 1}12"
recent = []
for sym, g in fc[fc["forecast_date"] == FY_NEXT].groupby("symbol"):
    now, n1 = consensus(g, asof)
    prev, n0 = consensus(g, asof - timedelta(days=LOOK))
    if n1 >= 2 and n0 >= 2 and prev > 0:
        recent.append({"symbol": sym, "최근 리비전": now / prev - 1, "증권사": n1})
recent = pd.DataFrame(recent).merge(uni, on="symbol").sort_values("최근 리비전", ascending=False)
picks = recent[recent["최근 리비전"] > 0].head(PICK)
print(f"■ {FY_NEXT[:4]}년 영업이익 1개월 리비전 상향 상위 {len(picks)}종목")
picks.assign(**{"최근 리비전": (picks["최근 리비전"] * 100).round(1)})[["name", "최근 리비전", "증권사"]]

■ 2027년 영업이익 1개월 리비전 상향 상위 8종목


,name,최근 리비전,증권사
12,HMM,48.5,3
18,삼성생명,16.1,3
28,SK이노베이션,14.2,5
8,삼성전기,11.9,5
1,SK하이닉스,6.1,5
17,삼성물산,4.4,5
6,삼성전자,4.1,7
25,셀트리온,3.9,7


## 3. 과거 상향 이벤트 찾기
거래일마다 **그해 결산(FY = 그 날짜의 연도)** 컨센서스를 계산하고, 전 거래일보다 3% 이상 오른 날을 이벤트로 잡습니다.
같은 연도끼리만 비교하므로 연초에 대상 연도가 바뀌는 날은 이벤트가 되지 않습니다.

In [6]:
prices = stock_prices(list(picks["symbol"]), hist_start, ASOF, columns=["close"])
px = prices.pivot_table(index="date", columns="symbol", values="close").sort_index()
kospi = ds_df(f"GetMarketIndexes(KRX:KOSPI, date_from={hist_start}, date_to={ASOF})").set_index("date")["close"]
days = px.index[px.index >= asof - pd.DateOffset(years=HIST_YEARS)]

events = []
for sym in picks["symbol"]:
    g = fc[fc["symbol"] == sym]
    prev_val, prev_fy, last_pos = None, None, -10 ** 9
    for pos, t in enumerate(days):
        fy = f"{t.year}12"
        val, n = consensus(g[g["forecast_date"] == fy], t)
        if prev_fy == fy and prev_val and prev_val > 0 and n >= 2 and val / prev_val - 1 >= JUMP and pos - last_pos >= GAP:
            events.append({"symbol": sym, "date": t, "FY": fy, "상향폭": val / prev_val - 1, "증권사": n})
            last_pos = pos
        prev_val, prev_fy = val, fy
ev = pd.DataFrame(events).merge(uni, on="symbol")
print(f"이벤트 {len(ev)}건")
ev.groupby("name").size().rename("이벤트 수").to_frame().T

이벤트 45건


name,HMM,SK이노베이션,SK하이닉스,삼성물산,삼성생명,삼성전기,삼성전자,셀트리온
이벤트 수,5,7,11,4,5,5,7,1


## 4. 이벤트 뒤 수익률
이벤트일 종가 → +N거래일 종가. 괄호 안은 같은 구간 코스피 대비 초과수익 평균입니다. 아직 N일이 지나지 않은 이벤트는 그 지평선에서 빠집니다.

In [7]:
k = kospi.reindex(px.index).ffill()
rows = []
for _, e in ev.iterrows():
    s = px[e["symbol"]].dropna()
    i = s.index.get_loc(e["date"])
    r = {"name": e["name"], "date": e["date"].date(), "상향폭": e["상향폭"]}
    for h in HORIZONS:
        if i + h < len(s):
            r[f"+{h}D"] = s.iloc[i + h] / s.iloc[i] - 1
            r[f"+{h}D 초과"] = r[f"+{h}D"] - (k.loc[s.index[i + h]] / k.loc[s.index[i]] - 1)
    rows.append(r)
res = pd.DataFrame(rows)

def summary(g):
    out = {}
    for h in HORIZONS:
        c = g[f"+{h}D"].dropna()
        x = g[f"+{h}D 초과"].dropna()
        out[f"+{h}D"] = f"{c.mean() * 100:+.1f}% ({len(c)}건, 승률 {(c > 0).mean() * 100:.0f}%, 초과 {x.mean() * 100:+.1f}%)" if len(c) else "-"
    return pd.Series(out)

tbl = res.groupby("name").apply(summary, include_groups=False)
tbl = picks.set_index("name")[["최근 리비전"]].join(tbl)
tbl["최근 리비전"] = (tbl["최근 리비전"] * 100).map(lambda x: f"{x:+.1f}%")
tbl

,최근 리비전,+1D,+5D,+20D
name,,,,
HMM,+48.5%,"+0.5% (5건, 승률 40%, 초과 +3.7%)","+0.1% (5건, 승률 60%, 초과 +5.8%)","-1.4% (4건, 승률 50%, 초과 -5.0%)"
삼성생명,+16.1%,"+1.1% (5건, 승률 80%, 초과 +1.9%)","+3.8% (5건, 승률 60%, 초과 +5.7%)","+14.2% (4건, 승률 50%, 초과 +10.0%)"
SK이노베이션,+14.2%,"-1.2% (7건, 승률 43%, 초과 +1.0%)","+0.8% (7건, 승률 57%, 초과 +2.8%)","+10.7% (6건, 승률 83%, 초과 +10.9%)"
삼성전기,+11.9%,"+1.4% (5건, 승률 80%, 초과 +2.2%)","+3.0% (5건, 승률 80%, 초과 +4.4%)","+53.0% (4건, 승률 100%, 초과 +38.6%)"
SK하이닉스,+6.1%,"-0.4% (11건, 승률 45%, 초과 -0.0%)","-0.1% (11건, 승률 45%, 초과 -0.6%)","+5.1% (10건, 승률 50%, 초과 +2.5%)"
삼성물산,+4.4%,"+3.6% (4건, 승률 25%, 초과 -0.5%)","+0.7% (4건, 승률 50%, 초과 +1.2%)","-2.6% (3건, 승률 33%, 초과 -3.0%)"
삼성전자,+4.1%,"+0.7% (7건, 승률 71%, 초과 -0.5%)","+3.0% (7건, 승률 71%, 초과 +1.3%)","+12.5% (7건, 승률 71%, 초과 +2.5%)"
셀트리온,+3.9%,"+1.2% (1건, 승률 100%, 초과 -1.3%)","+1.6% (1건, 승률 100%, 초과 +8.0%)","+15.1% (1건, 승률 100%, 초과 +29.3%)"


## 5. 한 종목 이벤트 상세
표에서 궁금한 종목의 이벤트를 날짜별로 봅니다.

In [8]:
DETAIL = picks["name"].iloc[0]
d = res[res["name"] == DETAIL].copy()
for c in ["상향폭"] + [f"+{h}D" for h in HORIZONS]:
    d[c] = (d[c] * 100).round(1)
d[["date", "상향폭"] + [f"+{h}D" for h in HORIZONS]]

,date,상향폭,+1D,+5D,+20D
0,2025-04-07,32.3,-1.4,5.3,-0.8
1,2025-07-15,3.9,-2.9,-6.2,-13.5
2,2026-05-14,43.3,-1.0,0.7,6.0
3,2026-07-23,16.3,3.1,-1.7,2.9
4,2026-10-29,9.6,4.7,2.2,NaN


## 결과 읽는 법과 한계
- `평균(건수, 승률, 초과)` 순입니다. **건수가 6건 이하면 통계로 보기 어렵습니다.**
- 이 기간(2025~2026)은 반도체 등 일부 업종이 크게 오른 시기라 절대수익보다 **코스피 대비 초과수익**을 같이 보세요.
- 컨센서스 이력은 DeepSearch가 받는 증권사(대형주 4~7곳)로 재구성한 것이라, 증권사 한 곳의 추정 변경이 3% 이벤트를 만들 수 있습니다.
- 이벤트일 종가 기준이라 장중에 나온 리포트는 이벤트일 수익률 일부를 놓칠 수 있습니다.